![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Processing and Database Design

*Project 1 — SQL: From Data to Insight*

**Team:**
**Dataset:**

---

Turning one flat file into a relational database: clean the data, split it into tables, wire up the keys, load it into SQLite.

> **Done when:** `data/project.db` exists with 3+ related tables, `sql/schema.sql` matches your ERD, no foreign key is dangling, and the row counts are what you expect.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the analysis, not whether you kept the scaffold.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


---
## 1. Load the raw data

The Global Impacts Dataset of Invasive Alien Species (GIDIAS) has first been published on the 21. of Mai 2025 by Springer Natur and authored by S. Bacher et al.  (https://doi.org/10.6084/m9.figshare.27908838)

In [83]:
df_raw = pd.read_csv("../data/raw/GIDIAS_20250417_machine_read.csv")

---
## 2. Clean

Work through the problems you listed in notebook 01. Every decision you make here — drop the nulls or keep them, cast or discard — is one you may be asked to defend on Friday, so note why.

### 2.1 Discarding unecessary columns
Discarding all columns that contain free text, except `UniqueID`, as well as columns that caontain boolean values and are not necessary for the analysis.

Note: It would make more sense to standardise the column names first. However, since the column names are being compared with the names provided in the metadata, standardisation will be carried out later on.

In [84]:
df_meta = pd.read_excel("../data/raw/GIDIAS_metadata_20250507.xlsx")
free_text_columns = []
for column in df_raw:
    meta_row = df_meta.loc[df_meta["Variable"] == column] # finding right row the metadata for each column form the raw data   
    if meta_row["Number or range of unique entries"].iloc[0] == "Free text": # filtering the column for wich the number or range of unique entries is free text
        free_text_columns.append(column) # add the name of the column to the list of columns with free text
print(free_text_columns)
free_text_columns.remove("UniqueID") # remove "UniqueID" from the list of columns to remove


['UniqueID', 'IAS.Functional.Group', 'native.range.of.IAS', 'Text.excerpt', 'Methodology.details', 'Affected.native.species.Details', 'Affected.ecosystem.property', 'investigated.level.of.organization', 'details.NCP', 'magnitude.NCP', 'details.CWB', 'magnitude.CWB.details', 'IPLC']


In [85]:
# removing the columns with free text values 
df_clean = df_raw.drop(columns= free_text_columns)

In [86]:
# creating a list of columns with boolean type values to remove
boolean_columns = df_raw.select_dtypes(include="bool").columns
exclude = ["Island.k", "Protected.area.k", "global.extinction"] # these columns can be interesting for the analyses and will be kept
boolean_columns = boolean_columns.difference(exclude)
boolean_columns = boolean_columns.to_list()

In [87]:
df_clean = df_clean.drop(columns = boolean_columns)

### 2.2 Data quality corrections
Standardising the column names and correcting the data types

In [91]:
# using the column_names_std to standardise the column names 
from functions.cleaning import column_names_std
df_clean = column_names_std(df_clean) 
df_clean.nunique()

rowid                              22865
uniqueid                           22865
ias_species_name                    3779
verified_name_gbif_taxon            3353
gbif_scientificname_with_author     3204
genus                               2079
family                               761
order                                242
class                                 84
phylum                                36
kingdom                                8
ias_taxon                              4
reference                           6783
doi                                 1848
assessor                             106
year                                  87
year_of_impact                        92
type_of_source                        15
language                              16
region                                 5
country_location                     414
island                               368
island_k                               2
protected_area                       413
protected_area_k

In [92]:
# separating the suffix "id" from the word in front
df_clean.columns = df_clean.columns.str.replace(r"id$", "_id", regex=True) 

In [93]:
# correcting the data types to numeric
df_clean["year"] = pd.to_numeric(df_clean["year"].astype("Int64"))
df_clean["year_of_impact"] = pd.to_numeric(df_clean["year_of_impact"].astype("Int64"))
df_clean["magnitude_nature"] = pd.to_numeric(df_clean["magnitude_nature"].astype("Int64")) # converting first to numeric to drop the comma

In [94]:
# correcting the data type to string
df_clean["magnitude_nature"] = df_clean["magnitude_nature"].astype(str)

### 2.3. Discarding null values

As the dataset mostly contains categorical values, null values cannot be replaced by means or medians. For each case, it should be assessed whether the corresponding row should be discarded from the analysis. Rows containing null values in the columns `year`, `year_of_impact` and `region` may be excluded, as these values are essential for the analysis.

In [106]:
# making a list of all the rows with null values in the columns yaer, yaer of impact and region
nan_rows = df_clean[df_clean["region"].isna() & df_clean["year"].isna() & df_clean["year_of_impact"].isna()].index

In [107]:
df_clean = df_clean.drop(index = nan_rows)

---
## 3. Design your tables

Decide what your tables are and how they relate before you write any code. Draw the ERD (Excalidraw, draw.io), export it as an image, commit it, and link it from your README.

Three tables minimum, with real primary and foreign keys.

Species:                             
**species_id**                        
ias_species_name   
verified_name_gbif_taxon 
gbif_scientificname_with_author 
genus                               
family                               
order                                
class                                 
phylum                                
kingdom 
ias_taxon    

Impact:
**unique_id**
**species_id**
reference                           
doi                                 
assessor                             
year                                  
year_of_impact                        
type_of_source   
units_of_analysis_clean               
spatial_scale                          
affected_native_species_taxon          
mechanism_nature_clean                
direction_nature                      
direct_or_indirect_nature              
global_extinction                      
magnitude_nature                       
affected_ncp_clean                    
direction_ncp                          
affected_cwb_clean                    
direction_cwb                          
magnitude_cwb      
language   
protected_area                       
protected_area_k                       
realm                 

Location:
**country_id**
region                                 
country_location                     
island                               
island_k                               


row_id                             
unique_id                          
ias_species_name                    
verified_name_gbif_taxon            
gbif_scientificname_with_author     
genus                               
family                               
order                                
class                                 
phylum                                
kingdom                                
ias_taxon                              
reference                           
doi                                 
assessor                             
year                                  
year_of_impact                        
type_of_source                        
language                              
region                                 
country_location                     
island                               
island_k                               
protected_area                       
protected_area_k                       
realm                                  
units_of_analysis_clean               
spatial_scale                          
affected_native_species_taxon          
mechanism_nature_clean                
direction_nature                      
direct_or_indirect_nature              
global_extinction                      
magnitude_nature                       
affected_ncp_clean                    
direction_ncp                          
affected_cwb_clean                    
direction_cwb                          
magnitude_cwb                          


In [111]:
df_clean.nunique()

row_id                             22805
unique_id                          22805
ias_species_name                    3723
verified_name_gbif_taxon            3301
gbif_scientificname_with_author     3152
genus                               2047
family                               755
order                                242
class                                 84
phylum                                36
kingdom                                8
ias_taxon                              4
reference                           6782
doi                                 1848
assessor                             106
year                                  87
year_of_impact                        92
type_of_source                        15
language                              16
region                                 5
country_location                     414
island                               368
island_k                               2
protected_area                       413
protected_area_k

In [112]:
df_raw.groupby("Country.Location")["Region"].nunique().max()

np.int64(3)

In [113]:
df_raw.groupby("IAS.Species.Name")["UniqueID"].nunique().describe()

count    3779.000000
mean        6.050542
std        19.427250
min         1.000000
25%         1.000000
50%         1.000000
75%         4.000000
max       410.000000
Name: UniqueID, dtype: float64

In [114]:
df_raw[["UniqueID", "IAS.Species.Name"]].head(20)

,UniqueID,IAS.Species.Name
0,AF_Fish_crayfish_00001,Gambusia affinis
1,AF_Fish_crayfish_00002,Onchorhynchus mykiss
2,AF_Fish_crayfish_00003,Procambrus clarkii
3,AF_Fish_crayfish_00004,Procambrus clarkii
4,AF_Fish_crayfish_00005,Ctenopharyngodon idella
5,AF_Fish_crayfish_00006,Apiosoma piscicola
6,AF_Fish_crayfish_00007,Schyzocotyle acheilognathi
7,AF_Fish_crayfish_00008,Lernea cyprinacea
8,AF_Fish_crayfish_00009,Ichthyophthirius multifiliis
9,AF_Fish_crayfish_00010,Chilodonella piscicola


---
## 4. Build the tables

Split the cleaned data into the tables you designed.

---
## 5. Check the foreign keys

**Do not skip this.** A dangling foreign key is the failure that costs people their Wednesday: the database loads without an error, then your joins silently drop rows and every number is quietly wrong. Check every key value exists in its parent table before you load anything.

---
## 6. Create the database and load it

Write your `CREATE TABLE` statements in `sql/schema.sql`, then load the tables — parents before children, or the foreign keys will be rejected.

---
## 7. Verify

Row counts per table, and one join that returns the rows you expect. Then open `data/project.db` in DB Browser and look at it.